# 2. tokenizer2:

### Learning Objectives
By the end of this lesson, you should be able to:

* Build a production-ready BPE Tokenizer that correctly handles Unicode, whitespace normalization, and special tokens.
* Implement byte-level fallback so the Tokenizer can encode any input, including emojis, CJK text, and code, without generating unknown tokens.
* Use a pre-tokenization regex to split text at appropriate word, number, punctuation, and whitespace boundaries before executing BPE merges.
* Train a custom Tokenizer on a corpus and compare its compression ratio on multilingual text with `tiktoken`.
* Understand the role of Chat Templates in converting structured messages into Token IDs.
* Explain the differences between an educational Python implementation and a production-ready Tokenizer in terms of speed, accuracy, and reproducibility.
---
### What is the Problem?

The BPE Tokenizer from Lesson 01 worked on English text. Now test that same Tokenizer with Japanese text, emojis, or Python code containing a mix of tabs and spaces; part of the process will likely break or produce an unsuitable output.

The problem is not with the BPE algorithm itself; the problem is that the implementation is not yet complete. A production-ready Tokenizer must:

- Handle input at the byte level, independent of language;
- Normalize Unicode according to a defined policy before splitting text;
- Have special tokens that are never split or merged with other tokens;
- Combine pre-tokenization with subword splitting;
- Provide reliable, and ideally reversible, encode and decode capabilities;
- Be fast enough not to become a bottleneck in the training pipeline;
- Save vocabulary, merge rules, normalization, and special token configurations in a versioned format so results are reproducible.

The GPT-2 vocabulary contains 50,257 tokens, and Llama 3 uses a vocabulary of 128,256 tokens. For GPT-4 family models, tokenizers typically employ vocabularies on the scale of approximately 100,000 tokens; however, the exact number depends on the model and the encoding used.

These numbers do not belong to small toy examples. The merge tables for such vocabularies are trained on massive volumes of data. Beyond BPE itself, components such as normalization, pre-tokenization, special token handling, and chat template formatting separate a tokenizer limited to a "hello world" phrase from one suitable for extensive internet-scale data.

In this lesson, you will build and understand these very components and the logic behind them.

---

### Core Concept: The Full Pipeline

A production-ready Tokenizer is not just a single algorithm; it is a pipeline composed of several stages, each solving a different problem.

    A[Raw Text] --> B[Normalize] --> C[Pre-tokenize]--> D[BPE Merge]--> E[Special Tokens]--> F[Token IDs]





Packages

In [4]:
import re
import unicodedata
from collections import Counter
from typing import Dict, List, Tuple, Union

In [5]:
import regex

PATTERN = regex.compile(
    r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+",
    flags=regex.IGNORECASE,
)

def pre_tokenize(text: str) -> list[str]:
    return PATTERN.findall(text)


print(pre_tokenize("I don't code in Python 3.11!"))

['I', ' don', "'t", ' code', ' in', ' Python', ' 3', '.', '11', '!']


This code is responsible for pre-tokenization based on the standard GPT-2 pattern.

This algorithm splits the text into smaller chunks prior to BPE to prevent consecutive words or punctuation marks from merging:


In [6]:
# create pattern
# if have `regex`, use that but have not use `re`
try:
    import regex
    GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )
except ImportError:
    GPT2_PATTERN = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )



## part 1: `def pre_tokenize`

To implement the `pre_tokenize` method, we must use `findall` with `GPT2_PATTERN` to extract all segments matching the regex rules as a list of strings.


In [7]:
# part 1----------------------------------------------
def pre_tokenize(text: str) -> List[str]:
    """
    Split input text into initial word/symbol chunks using the GPT-2 regex pattern.

    Args:
        text (str): Raw input text string to pre-tokenize.

    Returns:
        List[str]: A list of string chunks matched by the pre-tokenization regex.
    """
    # TODO: Apply GPT2_PATTERN regex iterator over text to extract all chunk string matches
    return GPT2_PATTERN.findall(text)


In [8]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
print(f"split form:---------------------\n{(sample_1.split())}")
print(f"use function pre_tokenize:------\n{pre_tokenize(sample_1)}")


------------------/1/-------------------
split form:---------------------
['I', 'am', 'Mohsen', 'Mohebbi.', 'I', 'participated', 'in', 'the', 'Daneshkar', 'Artificial', 'Intelligence', 'course.']
use function pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']


## part 2: `def apply_merge`




In [29]:
# Part 2----------------------------------------------
def apply_merge(byte_seq: List[int], pair: Tuple[int, int], new_id: int) -> List[int]:
    """
    Replace consecutive occurrences of a specific pair of token IDs in a sequence with a new token ID.

    Args:
        byte_seq (List[int]): Current sequence of token IDs.
        pair (Tuple[int, int]): A tuple (first_id, second_id) representing the pair to merge.
        new_id (int): The new token ID assigned to the merged pair.

    Returns:
        List[int]: A new list of token IDs with target pairs merged.
    """
    # TODO: Iterate through byte_seq, find adjacent matching pairs, and replace them with new_id
    # if len byte<2 , NOT merge
    if len(byte_seq) < 2:
        return list(byte_seq)

    # make merge list
    merged: list[int] = []
    i = 0
    first, second = pair

    while i < len(byte_seq):
        # len of text is end? 
        if i < len(byte_seq) - 1 and byte_seq[i] == first and byte_seq[i + 1] == second:
            merged.append(new_id)
            i += 2  # go to next pair
        else:
            merged.append(byte_seq[i])
            i += 1

    return merged


In [ ]:
# manual test
print("/1/".center(40, '-'))
sample_1 = "I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course."
# use pre_tokenize
test_chunks = pre_tokenize(sample_1)
print(f"main text:-------------------\n{sample_1}")
print(f"first use pre_tokenize:------\n{test_chunks}")

print("/2/".center(40, '-'))
# we need encode
test_bytes = list(sample_1.encode("utf-8"))

if len(test_bytes) < 2:
    print("len byte < 2")


test_pair, test_new_id = (ord("M"), ord("o")), 999

merged: list[int] = []
i = 0
first, second = test_pair

while i < len(test_bytes):
    # len of text is end? 
    if i < len(test_bytes) - 1 and test_bytes[i] == first and test_bytes[i + 1] == second:
        merged.append(test_new_id)
        i += 2  # go to next pair
    else: # if we have 1 token and have not pair token
        merged.append(test_bytes[i])
        i += 1

print(f"merge is : \n{merged}")

print("/test_fucntion2/".center(40, '-'))
print(f"pair : {test_pair} and new_id : {test_new_id}")

test_apply_merge = apply_merge(test_bytes, test_pair, test_new_id)
print(f"def apply_merge --------------\n{test_apply_merge}")

------------------/1/-------------------
main text:-------------------
I am Mohsen Mohebbi. I participated in the Daneshkar Artificial Intelligence course.
first use pre_tokenize:------
['I', ' am', ' Mohsen', ' Mohebbi', '.', ' I', ' participated', ' in', ' the', ' Daneshkar', ' Artificial', ' Intelligence', ' course', '.']
------------------/2/-------------------
merge is : 
[73, 32, 97, 109, 32, 999, 104, 115, 101, 110, 32, 999, 104, 101, 98, 98, 105, 46, 32, 73, 32, 112, 97, 114, 116, 105, 99, 105, 112, 97, 116, 101, 100, 32, 105, 110, 32, 116, 104, 101, 32, 68, 97, 110, 101, 115, 104, 107, 97, 114, 32, 65, 114, 116, 105, 102, 105, 99, 105, 97, 108, 32, 73, 110, 116, 101, 108, 108, 105, 103, 101, 110, 99, 101, 32, 99, 111, 117, 114, 115, 101, 46]
------------/test_fucntion2/------------
pair : (77, 111) and new_id : 999
def apply_merge --------------
[73, 32, 97, 109, 32, 999, 104, 115, 101, 110, 32, 999, 104, 101, 98, 98, 105, 46, 32, 73, 32, 112, 97, 114, 116, 105, 99, 105, 112, 

## part 3: `class SpecialTokenHandler`

Manages registration and regex-based splitting of special tokens during tokenization

1. `def add_token`
   - Saving the string-to-ID mapping
   - Constructing a regex that ORs all registered tokens together, allowing us to split the text based on them.
   - Every time a new token is added, `self.pattern` is reconstructed. Using `re.escape` is crucial because tokens like `<|endoftext|>` contain regex reserved characters. This pattern will be used in the next method (`split_with_specials`) to locate the exact positions of special tokens in the raw text.

2. `def split_with_specials`



In [32]:
# Part 3---------------------------------------
class SpecialTokenHandler:
    """
    Manages registration and regex-based splitting of special tokens during tokenization.
    """

    def __init__(self) -> None:
        """Initialize empty special tokens mapping and pattern compiler."""
        self.special_tokens: Dict[str, int] = {}
        self.pattern: Union[re.Pattern, None] = None

    def add_token(self, token_str: str, token_id: int) -> None:
        """
        Register a special token and update the combined regular expression pattern.

        Args:
            token_str (str): The string representation of the special token (e.g., '<|end|>').
            token_id (int): The integer vocabulary ID assigned to the special token.

        Returns:
            None
        """
        # TODO: Store the token ID mapping and update the compiled regex pattern using re.escape
        # Save token and ID
        self.special_tokens[token_str] = token_id
        
        # Constructing the regex pattern for all registered special token `]`
        # Escaping special characters "<|end|>" ➜ "<\\|end\\|>"
        # `|` IS NOT `or` in special characters
        patterns = [re.escape(t) for t in self.special_tokens.keys()]
        
        # add `or` between special and next search that
        combined_pattern = "|".join(patterns)
        
        # fast to run
        # self.pattern is An object 
        self.pattern = re.compile(combined_pattern)

    def split_with_specials(self, text: str) -> List[Tuple[str, bool]]:
        """
        Segment text into a list of tuples containing text chunks and boolean flags indicating special tokens.

        Args:
            text (str): Input text that may contain special tokens.

        Returns:
            List[Tuple[str, bool]]: A list of tuples where each tuple is (substring, is_special_flag).
        """
        # TODO: Search text using pattern, split into standard text vs special token parts, and tag each part
        if not text:
            return []

        # if have NOT special characters return text
        if not self.pattern or not self.special_tokens:
            return [(text, False)]

        result: list[tuple[str, bool]] = []
        last_end = 0

        # find special characters and start and end
        for match in self.pattern.finditer(text):
            start, end = match.span()

            # between special characters have normal text
            if start > last_end:
                result.append((text[last_end:start], False))

            # add token
            result.append((match.group(), True))
            last_end = end

        # if next end special characters have text, add
        if last_end < len(text):
            result.append((text[last_end:], False))

        return result


In [43]:
# manual test
print("/1/".center(40, '-'))
test_Special = SpecialTokenHandler()
# add special characters
test_Special.add_token("<|endoftext|>", 50256) # special characters and token
test_Special.add_token("<|pad|>", 50257) # special characters and token

text_1 = "Hello world<|endoftext|>how are you?<|pad|>"
test_result = test_Special.split_with_specials(text_1)

print("test_result\n", test_result)


------------------/1/-------------------
test_result
 [('Hello world', False), ('<|endoftext|>', True), ('how are you?', False), ('<|pad|>', True)]


## part 4: `class ProductionTokenizer`

Byte-Pair Encoding (BPE) tokenizer supporting training, normalization, special tokens, encoding, and decoding

1. `__init__`
   - Initialize vocabulary, merges, special token handler, and next available token ID
2. `normalize`
3. `train`
4. `add_special_token`
5. `encode`
6. `decode`
7. `vocab_size`
8. `get_token_bytes`



In [46]:
# part 4-------------------------------------------
class ProductionTokenizer:
    """
    Byte-Pair Encoding (BPE) tokenizer supporting training, normalization, special tokens, encoding, and decoding.
    """

    def __init__(self) -> None:
        """Initialize vocabulary, merges, special token handler, and next available token ID."""
        self.merges: Dict[Tuple[int, int], int] = {}
        self.vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)} # [0, 255]
        # use class SpecialTokenHandler : Composition 
        self.special_handler: SpecialTokenHandler = SpecialTokenHandler()
        self.next_id: int = 256 # 255 +1 ...

    def normalize(self, text: str) -> str:
        """
        Normalize input text using Unicode NFKC normalization.

        Args:
            text (str): Raw input string.

        Returns:
            str: Unicode normalized string.
        """
        # TODO: Normalize input text using unicodedata NFKC standard
        # In the guide file:
        # Apply Unicode normalization such as NFKC, and if necessary, perform lowercasing or accent removal
        # but in todo say" `NFKC standard`
        # from package unicodedata use normalize
        # in doc `unicodedata.normalize` have `forms = ["NFC", "NFD", "NFKC", "NFKD"]`
        # https://docs.python.org/3/library/unicodedata.html
        form_norm = "NFKC"
        return unicodedata.normalize(form_norm, text)

    def train(self, text: str, num_merges: int) -> None:
        """
        Train the BPE tokenizer by identifying frequent adjacent pairs and iteratively merging them.

        Args:
            text (str): Corpus text used for training the tokenizer.
            num_merges (int): Number of BPE merge operations to execute.

        Returns:
            None
        """
        # TODO: Normalize and pre-tokenize corpus text into byte sequences
        # TODO: Iteratively count adjacent pair frequencies across all chunk byte sequences
        # TODO: Find the most frequent pair, create a new vocabulary entry, and record the merge rule
        # TODO: Replace the best pair in all chunk sequences using apply_merge
        # TODO: Normalize and pre-tokenize corpus text into byte sequences

        # use normalizer
        norm_text = self.normalize(text)

        # basic chunk
        # change to byte UTF-8 `[0, 255]`
        chunks = pre_tokenize(norm_text)
        chunk_ids: list[list[int]] = [list(chunk.encode("utf-8")) for chunk in chunks]

        # TODO: Iteratively count adjacent pair frequencies across all chunk byte sequences
        for _ in range(num_merges):
            # Pair Frequency Counting
            pair_counts: dict[tuple[int, int], int] = {}
            # make merge
            for ids in chunk_ids:
                for i in range(len(ids) - 1):
                    pair = (ids[i], ids[i + 1])
                    pair_counts[pair] = pair_counts.get(pair, 0) + 1

            #if len chunk < 2 stop
            if not pair_counts:
                break

            # TODO: Find the most frequent pair, create a new vocabulary entry, and record the merge rule
            best_pair = max(pair_counts, key=pair_counts.get)
            new_id = self.next_id
            self.next_id += 1

            # add to roll merge
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            # TODO: Replace the best pair in all chunk sequences using apply_merge
            chunk_ids = [apply_merge(ids, best_pair, new_id) for ids in chunk_ids]

    def add_special_token(self, token_str: str) -> int:
        """
        Register a new special token into the tokenizer vocabulary and special token handler.

        Args:
            token_str (str): Special token string (e.g., '<|begin|>').

        Returns:
            int: Assigned vocabulary integer ID for the special token.
        """
        # TODO: Allocate new_id, register special token with special_handler and add byte representation to vocab
        raise NotImplementedError("Implement this method")

    def encode(self, text: str) -> List[int]:
        """
        Encode raw text into a sequence of vocabulary token IDs using trained merges and special tokens.

        Args:
            text (str): Input text string to be tokenized.

        Returns:
            List[int]: List of encoded vocabulary token IDs.
        """
        # TODO: Normalize input text and split into standard text and special token segments
        # TODO: For standard text segments, apply pre-tokenization and convert chunks into byte sequences
        # TODO: Apply learned BPE merges sequentially to byte sequences and collect all output token IDs
        raise NotImplementedError("Implement this method")

    def decode(self, ids: List[int]) -> str:
        """
        Decode a list of token IDs back into a UTF-8 string.

        Args:
            ids (List[int]): List of integer token IDs.

        Returns:
            str: Decoded UTF-8 text string.
        """
        # TODO: Map token IDs back to byte representations using vocabulary and decode as UTF-8
        raise NotImplementedError("Implement this method")

    def vocab_size(self) -> int:
        """
        Get current total size of vocabulary including base bytes, merges, and special tokens.

        Returns:
            int: Number of total entries in vocabulary.
        """
        # TODO: Return total number of vocabulary items
        raise NotImplementedError("Implement this method")

    def get_token_bytes(self, token_id: int) -> bytes:
        """
        Retrieve underlying byte representation of a given token ID.

        Args:
            token_id (int): Token ID to look up.

        Returns:
            bytes: Byte sequence corresponding to token_id, or default placeholder if not found.
        """
        # TODO: Retrieve byte mapping from vocabulary dictionary for specified token_id
        raise NotImplementedError("Implement this method")


In [ ]:
#manual test
# https://pypi.org/project/pyunormalize/
from pyunormalize import normalize

text = "désaﬃliât"
print("Input\t" + " ".join([f"{ord(c):04X}" for c in text]))

# Apply each of the four forms
forms = ["NFC", "NFD", "NFKC", "NFKD"]
for form in forms:
    normalized_text = normalize(form, text)
    hex_repr = " ".join([f"{ord(c):04X}" for c in normalized_text])
    print(f"{form}\t{hex_repr}")

# Output:
# Input   0064 00E9 0073 0061 FB03 006C 0069 00E2 0074
# NFC     0064 00E9 0073 0061 FB03 006C 0069 00E2 0074
# NFD     0064 0065 0301 0073 0061 FB03 006C 0069 0061 0302 0074
# NFKC    0064 00E9 0073 0061 0066 0066 0069 006C 0069 00E2 0074
# NFKD    0064 0065 0301 0073 0061 0066 0066 0069 006C 0069 0061 0302 0074